# RealMe Studio — Colab launcher

Three cells. **Use a CPU runtime** (Runtime → Change runtime type → CPU): the
voice-over-slides and podcast workflows need no GPU, and a CPU runtime costs no
compute units.

One thing worth knowing before you start: Colab's terms of service list
*"creating deepfakes"* among prohibited activities, on paid tiers too, and also
prohibit "connecting to remote proxies" — which is what a Gradio `share=True`
link is. This notebook therefore does voice and slides only, and reaches the
app through Colab's own built-in port proxy rather than a tunnel. Any
talking-head work belongs on hardware you control.


## 1 · Install

In [ ]:
!apt-get -qq install -y ffmpeg poppler-utils espeak-ng > /dev/null 2>&1
!pip -q install pydantic pypdf fastapi 'uvicorn[standard]' python-multipart

# Point this at wherever you keep the app
APP = '/content/drive/MyDrive/ccsandbox/ProjectRealMe/03_App'
import sys, os
sys.path.insert(0, APP)
os.chdir(APP)
!python -m realme.cli doctor

## 2 · Keys and storage

Put `GEMINI_API_KEY` (free at aistudio.google.com) and, once you have cloned a
voice, `ELEVENLABS_API_KEY` + `REALME_ELEVEN_VOICE_ID` into Colab **Secrets**
(the key icon in the left sidebar). Mounting Drive matters: the Colab VM
filesystem is wiped when the runtime is recycled, so everything the app writes
should land in Drive.

In [ ]:
from google.colab import userdata, drive
import os
for k in ("GEMINI_API_KEY", "ELEVENLABS_API_KEY", "REALME_ELEVEN_VOICE_ID"):
    try:
        os.environ[k] = userdata.get(k); print("loaded", k)
    except Exception:
        print("not set:", k)

drive.mount('/content/drive')
os.environ['REALME_HOME'] = '/content/drive/MyDrive/RealMeStudio'
os.makedirs(os.environ['REALME_HOME'], exist_ok=True)
print('data dir:', os.environ['REALME_HOME'])

## 3 · Launch the Studio

This starts the server in the background and opens it in a new tab through
Colab's port proxy. Keep this tab open — closing it stops the runtime.

In [ ]:
import subprocess, time, os
proc = subprocess.Popen(['python', '-m', 'realme.cli', 'studio', '--port', '8000'],
                        env={**os.environ, 'PYTHONPATH': APP})
time.sleep(6)
from google.colab import output
output.serve_kernel_port_as_window(8000)

### If you would rather use the command line

```bash
!python -m realme.cli lecture my_deck.pdf -o "$REALME_HOME" \
    --script gemini --tts elevenlabs \
    --context "PUBH 6002, second-year MPH. They know DAGs but not IPTW." \
    --style   "Dry and precise. Worked examples over abstraction."
```
